# Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import seaborn as sns
import missingno as msno
import plotly.express as px

from tqdm import tqdm

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error, make_scorer

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
import optuna

from tqdm import tqdm

import random
random.seed(42)
np.random.seed(42)

from features_eng import create_lag, full_df, correct_values


# Data loading 

In [ ]:
# Data Loading
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Feature Engineering (Features creation + cleaning)

In [ ]:
train = correct_values(train)
train = create_lag(train, 365)
train = full_df(train) 

In [ ]:
train.shape

# Cross validation

In [ ]:
# Adjust max_train_size or n_splits as needed
tscv = TimeSeriesSplit(n_splits=5, max_train_size=20000, test_size=5000, gap=100)

for train_index, test_index in tscv.split(train):
    print("TRAIN:", len(train_index), "TEST:", len(test_index))
    print("TRAIN:", train_index, "TEST:", test_index)
    print('***********')
    

In [ ]:
fig, axs = plt.subplots(5, 1, figsize = (10, 20), sharex = True)

fold = 0
for train_index, test_index in tscv.split(train):
    train_data = train.iloc[train_index]
    test_data = train.iloc[test_index]
    train_data['GT_NO2'].plot(ax = axs[fold],
                               label = 'Train',
                               title = 'Train and Test Data')
    test_data['GT_NO2'].plot(ax = axs[fold],
                               label = 'Test',
                               title = 'Train and Test Data')
    fold += 1
plt.show()

# Data preparation

In [ ]:
train = train.dropna(subset=['GT_NO2'])
X = train.drop(columns=['GT_NO2'])
y = train['GT_NO2']

# Metric

In [ ]:
rmse_scorer = make_scorer(root_mean_squared_error)

# XG Boost

In [ ]:
xgb = XGBRegressor(base_score=0.5, booster='gbtree',
                    n_estimators=1000,
                    #early_stopping_rounds=50,
                    objective='reg:linear',
                    max_depth=3,
                    learning_rate=0.01)

In [ ]:
%%time
cv_scores = cross_val_score(xgb, X, y, cv=tscv, scoring=rmse_scorer)

# Print the RMSE scores for each fold
print("Cross-validated RMSE scores:", cv_scores)
print("Mean RMSE:", np.mean(cv_scores))
print("Standard Deviation of RMSE:", np.std(cv_scores))

## Hyper-parameter tunning (Too expensive)

def objective(trial):
    params = {
    'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
    'max_depth': trial.suggest_int('max_depth', 3, 50),
    'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
    'subsample': trial.suggest_float('subsample', 0.5, 1.0),
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
    'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)
    }

    xgb = XGBRegressor(**params)
    score = cross_val_score(xgb, X, y, cv=tscv, scoring=rmse_scorer, n_jobs=-1).mean()
    return score

study = optuna.create_study(direction='minimize',
                            sampler = optuna.samplers.RandomSampler(seed=42))
study.optimize(objective, n_trials=500)

print("Best trial:")
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

# LightGBM

In [ ]:
lgbm = LGBMRegressor(boosting_type='gbdt',
                     objective='regression',
                     num_leaves=30,
                     learning_rate=0.05,
                     n_estimators=100)

rmse_scorer = make_scorer(root_mean_squared_error)

In [ ]:
%%time
cv_scores = cross_val_score(lgbm, X, y, cv=tscv, scoring=rmse_scorer, n_jobs=-1)

# Print the RMSE scores for each fold
print("Cross-validated RMSE scores:", cv_scores)
print("Mean RMSE:", np.mean(cv_scores))
print("Standard Deviation of RMSE:", np.std(cv_scores))

## Hyper-parameter tunning

In [ ]:
# 1. Define an objective function to be maximized.
def objective(trial):
    ...

    # 2. Suggest values of the hyperparameters using a trial object.
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }

    lgbm = LGBMRegressor(**param)
    score = cross_val_score(lgbm, X, y, cv=tscv, scoring=rmse_scorer, n_jobs=-1).mean()
    return score

In [ ]:
%%time

study = optuna.create_study(direction='minimize',
                            sampler = optuna.samplers.RandomSampler(seed=42))
study.optimize(objective, n_trials=20)

print("Best trial:")
trial = study.best_trial
print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

## Visualization

In [ ]:
study.best_params

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(study)

In [ ]:
optuna.visualization.plot_slice(study, params=['lambda_l1', 'lambda_l2', 'num_leaves', 'feature_fraction', 'bagging_fraction', 'bagging_freq', 'min_child_samples' ])

In [ ]:
optuna.visualization.plot_param_importances(study)

# Cat Boost